<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_0_Tokens_and_Embeddings/23_0_2_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 2
## Embeddings

---

## What This Notebook Is About

23_0_1 turned text into token IDs. An ID is just a label, though: ID 3,185 isn't "close to" ID 3,186 in any useful sense, so a network can't do arithmetic with IDs directly. The first layer of every language model replaces each ID with a vector of numbers called an **embedding**. The vectors are learned during training, and they end up placing tokens that are used in similar ways near each other.

This is the same idea as the 512 features in 19_0_2, which placed similar photos near each other. Here it's tokens, and later whole reviews.

**What you will learn:**
1. How an **embedding layer** turns IDs into vectors (it's a lookup table, and also a matrix multiplication)
2. How to measure whether two embeddings are similar, using the dot product from 19_1_3
3. What DistilBERT's learned embeddings put next to each other, and one surprising thing about it
4. Why one vector per token isn't enough, since the same word can mean different things
5. How to embed whole reviews, and use the embeddings to find similar reviews

This notebook downloads two pretrained models from Hugging Face (about 350 MB together). It doesn't need a GPU; the slowest cell takes under a minute on a CPU.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

reviews = pd.read_csv(fetch('imdb_reviews.csv'))
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
distilbert = AutoModel.from_pretrained('distilbert-base-uncased').eval()

(Loading DistilBERT prints a report listing a few "unexpected" weights. They belong to a part of the model used only during its original training, which we don't need, and the message can be ignored.)

---

## Section 1: An Embedding Layer Is a Lookup Table

`nn.Embedding(vocabulary_size, dimension)` holds one vector of length `dimension` for every ID in the vocabulary, as the rows of a weight matrix. Given IDs, it returns their rows. Here is a tiny one with a vocabulary of 5 and vectors of length 3:

In [ ]:
torch.manual_seed(0)
embedding = nn.Embedding(5, 3)
print('weight (one row per ID):')
print(embedding.weight.data)

ids = torch.tensor([4, 0, 4])
print('\nembedding of IDs 4, 0, 4:')
print(embedding(ids).data)

The result has one row per ID, copied from the weight matrix: rows 4, 0 and 4 again. The same token always gets the same vector.

Looking up a row is the same as a matrix multiplication. Write each ID as a **one-hot** vector, all zeros except a 1 in the ID's position, and multiply it by the weight matrix. The 1 picks out that row, and every other row is multiplied by 0:

In [ ]:
one_hot = nn.functional.one_hot(ids, num_classes=5).float()
print(one_hot)
print(torch.allclose(one_hot @ embedding.weight, embedding(ids)))

So an embedding layer is an `nn.Linear` layer without a bias, applied to one-hot vectors, and stored as a lookup because that's much faster than multiplying by tens of thousands of zeros. Its weights are trained by gradient descent like any other layer's. They start random, and during training each token's vector is pushed toward whatever values help the model with its task.

---

## Section 2: DistilBERT's Embeddings

DistilBERT's first layer is an embedding table for its 30,522 tokens, each with a vector of 768 numbers:

In [ ]:
table = distilbert.embeddings.word_embeddings.weight.detach()
print('embedding table:', tuple(table.shape))
print(f'{table.numel():,} of DistilBERT\'s {sum(p.numel() for p in distilbert.parameters()):,} weights')

movie_id = tokenizer.convert_tokens_to_ids('movie')
print(f'\nID of "movie": {movie_id}; first 8 numbers of its vector: {table[movie_id, :8].numpy().round(3)}')

About a third of DistilBERT's weights are this one table. Like ResNet's 512 features, the individual numbers don't mean anything we can name. What matters is how the vectors relate to each other.

### Measuring similarity

19_1_3 introduced the dot product: multiply matching elements and add. Two vectors that point in a similar direction have a large positive dot product, and two that point in unrelated directions have a dot product near 0. To compare directions without the vectors' lengths getting in the way, first scale every vector to length 1. The dot product of two length-1 vectors is called their **cosine similarity**. It's 1 for vectors pointing the same way, 0 for unrelated ones and −1 for opposite ones.

In [ ]:
unit = table / table.norm(dim=1, keepdim=True)        # every vector scaled to length 1

def similarity(a, b):
    return (unit[tokenizer.convert_tokens_to_ids(a)] @ unit[tokenizer.convert_tokens_to_ids(b)]).item()

for a, b in [('movie', 'film'), ('movie', 'banana'), ('actor', 'actress'), ('good', 'great')]:
    print(f'{a:>6} / {b:<8} {similarity(a, b):.3f}')

`table.norm(dim=1, keepdim=True)` is the length of each row (the square root of the sum of its squared numbers), so dividing by it scales every row to length 1.

Even "movie" and "banana" have a similarity of about 0.4, not 0. In these learned embeddings every vector shares some common direction, so similarities are best compared with each other ("film" is much closer to "movie" than "banana" is) rather than read against 0.

### Nearest neighbors

The dot product of one token's vector with the whole table gives its similarity to every token at once, as in 19_1_3's matrix multiplication. The highest values are the token's **nearest neighbors**:

In [ ]:
def neighbors(word, k=6):
    sims = unit @ unit[tokenizer.convert_tokens_to_ids(word)]      # (30522,) similarities
    best = sims.topk(k + 1).indices[1:]                             # skip the word itself
    return [tokenizer.convert_ids_to_tokens(int(i)) for i in best]

for word in ['movie', 'actor', 'paris', 'funny', 'boring', 'good', 'terrible']:
    print(f'{word:9} {neighbors(word)}')

Most neighbors make sense: plurals and synonyms ("movie": "movies", "film", "flick"), related roles ("actor": "actress"), related places ("paris": "france", "london"). Nobody told DistilBERT any of this. It was trained only to predict missing words in a huge amount of English text, and words that can fill the same gaps ended up with similar vectors.

That also explains the surprise in the last two rows. Among the nearest neighbors of "good" is "bad", and among those of "terrible" is "terrific". Opposites are used in exactly the same places ("the movie was ___"), so a model that learns from where words appear puts them close together. These embeddings capture how words are used, which is related to what they mean but not the same thing. For a sentiment classifier that matters: "good" and "bad" start out close, and the later layers of the model have to learn to separate them.

### A picture

The vectors have 768 numbers, so they can't be plotted directly. **Principal component analysis** (PCA) finds the two directions along which a set of vectors differs the most, and plots each vector's position along those two directions. Here are some words from four groups:

In [ ]:
groups = {
    'praise': ['excellent', 'wonderful', 'brilliant', 'great', 'superb'],
    'criticism': ['awful', 'terrible', 'boring', 'dull', 'worst'],
    'genres': ['horror', 'comedy', 'western', 'documentary', 'musical'],
    'people': ['actor', 'director', 'writer', 'actress', 'producer'],
}
words = [w for ws in groups.values() for w in ws]
vectors = unit[[tokenizer.convert_tokens_to_ids(w) for w in words]]

centered = vectors - vectors.mean(dim=0)
_, _, directions = torch.pca_lowrank(centered, q=2, center=False)
points = (centered @ directions).numpy()

plt.figure(figsize=(7, 5.5))
i = 0
for name, ws in groups.items():
    plt.scatter(points[i:i + len(ws), 0], points[i:i + len(ws), 1], label=name, s=50)
    for w, (x, y) in zip(ws, points[i:i + len(ws)]):
        plt.annotate(w, (x, y), xytext=(4, 4), textcoords='offset points', fontsize=9)
    i += len(ws)
plt.legend()
plt.title("DistilBERT token embeddings, projected to 2 dimensions")
plt.show()

The words for people sit apart from the rest. The praise and criticism words are thoroughly mixed together, which is the "good"/"bad" problem again: in where they're used, "awful" and "wonderful" are very alike. The genres are spread out, and "western" ends up far from the other genres, next to "great". Two dimensions can only show a small part of 768, so a picture like this is a rough guide to how the vectors are arranged, not a measurement.

---

## Section 3: One Vector per Token Isn't Enough

The embedding table gives "bank" the same vector whatever sentence it's in. But "bank" means different things in "the bank approved my loan" and "we had a picnic on the bank of the river". A language model needs vectors that depend on context.

That is what the rest of DistilBERT does. Its six layers after the embedding table repeatedly update each token's vector using the tokens around it, and the final vectors are **contextual embeddings**. Compare the vector for "bank" in three sentences, before and after those layers:

In [ ]:
def bank_vectors(sentence):
    encoded = tokenizer(sentence, return_tensors='pt')
    position = encoded['input_ids'][0].tolist().index(tokenizer.convert_tokens_to_ids('bank'))
    with torch.no_grad():
        final = distilbert(**encoded).last_hidden_state[0, position]      # after all six layers
    first = table[encoded['input_ids'][0, position]]                      # straight from the table
    return first, final

sentences = {'money': 'I deposited my paycheck at the bank.',
             'loan': 'The bank approved my loan.',
             'river': 'We had a picnic on the bank of the river.'}
vecs = {k: bank_vectors(s) for k, s in sentences.items()}
cos = lambda a, b: nn.functional.cosine_similarity(a, b, dim=0).item()

print('                  from the table   after DistilBERT')
for a, b in [('money', 'loan'), ('money', 'river')]:
    print(f'{a:>6} vs {b:<6}       {cos(vecs[a][0], vecs[b][0]):.3f}            {cos(vecs[a][1], vecs[b][1]):.3f}')

From the table, "bank" is identical in every sentence (a similarity of 1). After DistilBERT's layers, the two money meanings are much more similar to each other than either is to the river meaning. Each token's vector has taken on information from the words around it. The mechanism that lets a token's vector draw on the other tokens in the text is **attention**, the subject of 23_1.

---

## Section 4: Embeddings for Whole Reviews

The same idea extends from tokens to whole texts. A **sentence embedding** model turns a whole piece of text into one vector, trained so that texts with similar meanings get similar vectors. `all-MiniLM-L6-v2` is a small, widely used one. It runs the text through a transformer like DistilBERT and averages the contextual vectors of its tokens (skipping padding, using the attention mask from 23_0_1) into a single vector of 384 numbers.

In [ ]:
st_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
st_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').eval()

def embed(texts, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = st_tokenizer(texts[i:i + batch_size], truncation=True, max_length=256,
                             padding=True, return_tensors='pt')
        with torch.no_grad():
            tokens = st_model(**batch).last_hidden_state              # (batch, tokens, 384)
        mask = batch['attention_mask'].unsqueeze(-1)                  # 1 for real tokens, 0 for padding
        mean = (tokens * mask).sum(dim=1) / mask.sum(dim=1)           # average of the real tokens only
        out.append(nn.functional.normalize(mean, dim=1))              # length 1, for cosine similarity
    return torch.cat(out)

search_set = reviews[reviews['split'] == 'test'].iloc[:500]
review_vectors = embed(list(search_set['text']))
print('review embeddings:', tuple(review_vectors.shape))

Now each of 500 test reviews is a vector of 384 numbers. To search them, embed a description of what you're looking for and find the reviews with the highest cosine similarity. No review needs to contain the words of the query.

In [ ]:
def search(query, k=3):
    sims = review_vectors @ embed([query])[0]
    print(f'QUERY: {query}')
    for i in sims.topk(k).indices:
        row = search_set.iloc[int(i)]
        print(f"  [{sims[i]:.2f}, {'positive' if row['label'] else 'negative'}] {row['text'][:220]}...")
    print()

search('a horror film that was not scary at all')
search('beautiful animation that children and parents will both enjoy')
search('great special effects but the story made no sense')

The matches are useful but uneven. The horror query found reviews of horror films, including one complaining that the film is "not even creepy". The animation query found a review of a Miyazaki film and other family films. The third query found reviews complaining that a film made no sense, but none of them is about special effects: one vector of 384 numbers summarizes a whole review, and for this query the "made no sense" part of the meaning won out. The number in brackets, the similarity, is a better guide to how close a match is than the rank. Search engines, recommendation systems, and the systems that let chatbots look things up in a set of documents all work on this principle: embed everything once, then find the nearest vectors.

Whether these review vectors are good for telling positive reviews from negative ones is a different question. This model was trained to capture what texts are *about*, and two reviews of the same film can be about the same thing and still disagree. 23_2 compares several ways of classifying sentiment, including this one.

---

## Summary

- An **embedding layer** stores one learned vector per token ID and looks it up. It's the same as multiplying a **one-hot** vector by a weight matrix, and it's trained by gradient descent like any layer.
- DistilBERT's embedding table holds a 768-number vector for each of its 30,522 tokens, about a third of its weights.
- **Cosine similarity**, the dot product of two vectors scaled to length 1, measures how similar two embeddings are. Multiplying by the whole table gives a token's **nearest neighbors**.
- Embeddings learned from where words appear put synonyms together, but also opposites like "good" and "bad", because opposites are used in the same places.
- The table gives a word the same vector in every sentence. A transformer's layers produce **contextual embeddings** that depend on the surrounding words, so "bank" by a river and "bank" with a loan end up far apart. The mechanism is attention (23_1).
- **Sentence embeddings** turn whole texts into vectors where similar texts are close, which makes search by meaning possible.

**Next:** 23_1 works out attention by hand: how each token's vector becomes a weighted average of the others, with the weights coming from dot products and softmax.